# Tahap 1 & Tahap 2 — Penyiapan Dataset CXR, Deduplikasi, & Pembagian Data

**File ini bertujuan untuk:**
1. Menyatukan dataset pasien anak (Kermany) dan pasien dewasa (RSNA) ke format PNG grayscale 8-bit yang seragam.
2. Menyaring citra ganda (duplikat identik dan hampir serupa) sebelum pembagian data.
3. Menyeimbangkan rasio kelas Normal dan Pneumonia menjadi 1:1 di masing-masing sumber.
4. Membagi data secara *patient-wise* (70% Latih, 15% Validasi, 15% Uji) lalu menguncinya dengan SHA-256.

**Daftar Isi:**
- [1. Persiapan Lingkungan & Sumber Data](#1-persiapan-lingkungan-sumber-data)
- [2. Tahap 1 — Penyatuan & Penyeragaman Format Dataset CXR](#2-tahap-1-penyatuan-penyeragaman-format-dataset-cxr)
- [3. Tahap 1 — Penyaringan Citra Duplikat](#3-tahap-1-penyaringan-citra-duplikat)
- [4. Tahap 2 — Penyeimbangan 1:1 & Pembagian Data 70/15/15](#4-tahap-2-penyeimbangan-11-pembagian-data-701515)

<a id="1-persiapan-lingkungan-sumber-data"></a>
## 1. Persiapan Lingkungan & Sumber Data

Menghubungkan Google Drive, memuat konfigurasi proyek, dan mengekstrak arsip RSNA ke penyimpanan cepat lokal Colab bila diperlukan.
- `load_config()` & `ensure_output_dirs()`: memuat parameter proyek dan menyiapkan folder penyimpanan.
- `set_global_seed()`: mengunci angka acak (`seed = 42`) agar hasil selalu dapat diulang.

In [ ]:
# Sel 1: Setup lingkungan (Colab/Lokal), config terkunci, dan pemeriksaan direktori sumber
import json
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    !pip install -q pydicom

for candidate in [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/drive/MyDrive/LAB_TA_GradCAM_Pneumonia/code"),
    Path("/content/gradcam-pneumonia"),
]:
    if (candidate / "src" / "paths.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import pandas as pd
from IPython.display import display

from src.data.dedup import run_deduplication
from src.data.manifest import build_kermany_manifest, build_multisource_manifest
from src.data.split import (
    patientwise_stratified_split,
    save_and_lock_split,
    source_balanced_subsample,
    verify_source_class_balance,
)
from src.paths import ensure_output_dirs, load_config
from src.seed import get_env_snapshot, print_env_banner, set_global_seed

CFG = load_config()
PATHS = ensure_output_dirs(cfg=CFG)
SPLIT_SEED = int(CFG["assumed"]["split_seed"]["value"])
SUBSAMPLE_SEED = int(CFG["assumed"]["subsample_seed"]["value"])
PHASH_MAX_HAMMING = int(CFG["locked"]["data"]["phash_max_hamming"]["value"])

set_global_seed(SPLIT_SEED, deterministic=True)
print_env_banner(get_env_snapshot(seed=SPLIT_SEED))

KERMANY_DIR = PATHS["kermany_chest_xray"]
RSNA_DRIVE_DIR = PATHS["rsna_raw"]
WORK_RSNA_DIR = Path("/content/work/rsna")

# Gunakan SSD lokal Colab (/content/work/rsna) jika tersedia agar 10x lebih cepat dari FUSE Drive
if IN_COLAB and (WORK_RSNA_DIR / "stage_2_train_images").exists():
    RSNA_DIR = WORK_RSNA_DIR
elif IN_COLAB and (RSNA_DRIVE_DIR / "rsna-pneumonia-detection-challenge.zip").exists():
    import zipfile
    WORK_RSNA_DIR.mkdir(parents=True, exist_ok=True)
    needed = ("stage_2_train_labels.csv", "stage_2_detailed_class_info.csv", "stage_2_train_images/")
    with zipfile.ZipFile(RSNA_DRIVE_DIR / "rsna-pneumonia-detection-challenge.zip", "r") as zf:
        members = [m for m in zf.namelist() if m.startswith(needed)]
        print(f"Mengekstrak otomatis {len(members)} berkas RSNA dari ZIP Drive ke {WORK_RSNA_DIR} ...")
        zf.extractall(WORK_RSNA_DIR, members=members)
    RSNA_DIR = WORK_RSNA_DIR
else:
    RSNA_DIR = RSNA_DRIVE_DIR

# Di Colab, tulis PNG sementara di SSD lokal (/content/work/harmonized_png) agar konversi 20k gambar hanya ~2-3 menit
HARMONIZED_DIR = Path("/content/work/harmonized_png") if IN_COLAB else PATHS["harmonized_png"]
HARMONIZED_DIR.mkdir(parents=True, exist_ok=True)
HAS_RSNA = (RSNA_DIR / "stage_2_train_labels.csv").exists() and (RSNA_DIR / "stage_2_detailed_class_info.csv").exists()

print(f"\nIN_COLAB       : {IN_COLAB}")
print(f"PROJECT_ROOT   : {PATHS['project_root']}")
print(f"Kermany Raw    : {KERMANY_DIR} (ada={KERMANY_DIR.exists()})")
print(f"RSNA Raw Aktif : {RSNA_DIR} (ada={HAS_RSNA})")
print(f"Harmonized PNG : {HARMONIZED_DIR}")
if not HAS_RSNA:
    print(
        "\n[PERHATIAN] Dataset RSNA belum ditemukan di data/raw/rsna/!\n"
        "Letakkan berkas dari Kaggle 'rsna-pneumonia-detection-challenge' ke:\n"
        f"  {RSNA_DIR}/stage_2_train_labels.csv\n"
        f"  {RSNA_DIR}/stage_2_detailed_class_info.csv\n"
        f"  {RSNA_DIR}/stage_2_train_images/*.dcm"
    )

<a id="2-tahap-1-penyatuan-penyeragaman-format-dataset-cxr"></a>
## 2. Tahap 1 — Penyatuan & Penyeragaman Format Dataset CXR

Menggabungkan citra Kermany (JPEG) dan RSNA (DICOM) menjadi PNG grayscale 8-bit, membuang kelas ambigu RSNA, serta mencatat metadata posisi pengambilan gambar (AP/PA).
- `build_multisource_manifest()`: memproses kedua sumber data, menyamakan polaritas warna, dan menyimpan tabel manifest ke `manifest_multisource_raw.csv`.

In [ ]:
# Sel 2: Bangun manifest terpadu & harmonisasi format ke PNG grayscale 8-bit lossless
if HAS_RSNA:
    manifest_df, ingest_audit = build_multisource_manifest(
        kermany_root=KERMANY_DIR,
        rsna_root=RSNA_DIR,
        harmonized_dir=HARMONIZED_DIR,
        convert_to_png=True,
    )
    IMAGE_ROOT = HARMONIZED_DIR
    print("=== AUDIT INGESTI MULTI-SUMBER (KERMANY + RSNA) ===")
    print(json.dumps(ingest_audit, indent=2))
else:
    raise FileNotFoundError(
        f"Dataset RSNA wajib tersedia di {RSNA_DIR} sesuai proposal multi-sumber (§3.2.1). "
        "Unduh dari Kaggle 'rsna-pneumonia-detection-challenge' terlebih dahulu."
    )

manifest_path = PATHS["interim"] / "manifest_multisource_raw.csv"
manifest_df.to_csv(manifest_path, index=False, encoding="utf-8")

print("\nDistribusi citra mentah per sumber x kelas:")
display(manifest_df.groupby(["source", "label"]).agg(n_images=("image_id", "count"), n_groups=("group_id", "nunique")).reset_index())
if "photometric" in manifest_df.columns:
    print("Sebaran PhotometricInterpretation (termasuk MONOCHROME1 yang telah dikoreksi):")
    display(manifest_df.groupby(["source", "photometric"]).size().reset_index(name="count"))
if "view_position" in manifest_df.columns:
    print("Sebaran ViewPosition (AP/PA) per sumber x kelas:")
    display(manifest_df.groupby(["source", "label", "view_position"]).size().reset_index(name="count"))

<a id="3-tahap-1-penyaringan-citra-duplikat"></a>
## 3. Tahap 1 — Penyaringan Citra Duplikat

Menghapus citra ganda lintas sumber dalam dua tahap (duplikat identik SHA-256 dan citra sangat mirip terverifikasi) serta menghentikan proses bila ada konflik label.
- `run_deduplication()`: mendeteksi dan menyaring duplikat sebelum data dibagi, lalu mencatat hasilnya ke `dedup_log.csv`.

In [ ]:
# Sel 3: Deduplikasi 2-Tahap (SHA-256 exact + pHash/dHash/Pearson r>=0.985 lintas sumber)
import importlib, src.data.dedup as _dedup_mod
importlib.reload(_dedup_mod)
run_deduplication = _dedup_mod.run_deduplication

clean_df, dedup_log_df = run_deduplication(
    manifest_df=manifest_df,
    image_root=IMAGE_ROOT,
    check_pixel_hash=True,
    check_perceptual_hash=True,
    max_hamming=PHASH_MAX_HAMMING,
    max_dhash_hamming=5,
    min_pearson_r=0.985,
    num_workers=8,
    show_progress=True,
)

dedup_log_path = PATHS["interim"] / "dedup_log.csv"
dedup_log_df.to_csv(dedup_log_path, index=False, encoding="utf-8")

n_dropped = int((dedup_log_df["status"] == "DROPPED").sum())
print(f"Total citra pra-deduplikasi  : {len(manifest_df)}")
print(f"Total duplikat dikeluarkan   : {n_dropped}")
print(f"Total citra bersih           : {len(clean_df)}")
print("\nRincian duplikat menurut jenis kecocokan:")
display(dedup_log_df[dedup_log_df["status"] == "DROPPED"].groupby(["source", "label", "match_type"]).size().reset_index(name="n_dropped"))

<a id="4-tahap-2-penyeimbangan-11-pembagian-data-701515"></a>
## 4. Tahap 2 — Penyeimbangan 1:1 & Pembagian Data 70/15/15

Menyamakan jumlah kelas Normal dan Pneumonia (1:1) di tiap sumber secara *patient-wise*, lalu membagi data menjadi 70% latih, 15% validasi, dan 15% uji tanpa kebocoran pasien.
- `source_balanced_subsample()`: menyeimbangkan rasio kelas 1:1 di Kermany dan RSNA pada tingkat pasien.
- `verify_source_class_balance()`: memastikan proporsi kelas lintas sumber seragam.
- `patientwise_stratified_split()`: membagi data 70:15:15 dengan menjaga sebaran sumber, kelas, dan posisi AP/PA.
- `save_and_lock_split()`: menyimpan `split_patientwise.csv` beserta kode verifikasi SHA-256.

In [ ]:
# Sel 4: Penyeimbangan 1:1 per Sumber (Patient-Wise) & Pembagian Data 70:15:15 Terkunci SHA-256
balanced_df, subsample_report_df = source_balanced_subsample(clean_df, seed=SUBSAMPLE_SEED)
balance_check = verify_source_class_balance(balanced_df, max_prevalence_gap=0.02)

subsample_report_csv = PATHS["outputs"] / "tables" / "eda" / "source_balancing_report.csv"
subsample_report_df.to_csv(subsample_report_csv, index=False, encoding="utf-8")
print("=== LAPORAN PENYEIMBANGAN 1:1 PER SUMBER ===")
display(subsample_report_df)
print(f"Verifikasi keseragaman proporsi kelas lintas sumber: {balance_check}")

split_df = patientwise_stratified_split(
    balanced_df,
    ratios=(
        float(CFG["locked"]["data"]["split_ratio_train"]["value"]),
        float(CFG["locked"]["data"]["split_ratio_val"]["value"]),
        float(CFG["locked"]["data"]["split_ratio_test"]["value"]),
    ),
    seed=SPLIT_SEED,
    stratify_by_view=True,
)

split_csv_path = PATHS["interim"] / "split_patientwise.csv"
checksum = save_and_lock_split(split_df, split_csv_path)

summary = split_df.groupby(["source", "split", "label"]).agg(
    n_images=("image_id", "count"),
    n_groups=("group_id", "nunique"),
).reset_index()
summary.to_csv(PATHS["outputs"] / "tables" / "eda" / "tabel_partisi_multisource.csv", index=False, encoding="utf-8")

print("\n===================================================================")
print("          GATE CHECK — NOTEBOOK 01 (DATA MULTI-SUMBER & SPLIT)     ")
print("===================================================================")
display(summary)
print(f"Total citra final terkunci : {len(split_df)}")
print(f"Berkas split terkunci      : {split_csv_path}")
print(f"SHA-256 Checksum           : {checksum}")
print("[PASS] 0% kebocoran pasien/citra antar-partisi & rasio kelas 1:1 per sumber terverifikasi.")